# 02 — Preprocessing and feature extraction
Explore the shared image utilities. EXIF orientation and RGB conversion provide consistent decoding; resizing and normalization make CNN tensors consistent. Hand-engineered descriptors let classical models consume fixed-length numeric vectors.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing import IMAGE_SIZE, extract_image_features, image_to_array, iter_dataset_images, load_image
root = Path('../data/images') if not Path('data/images').exists() else Path('data/images')
try:
    example_path, example_label = next(iter(iter_dataset_images(root)))
    image = load_image(example_path)
    print('class label:', example_label, '(0 real, 1 synthetic)')
    print('original decoded dimensions:', image.size)
except (FileNotFoundError, StopIteration) as exc:
    image = None
    print(f'Add local data first: {exc}')

In [ ]:
if image is not None:
    tensor = image_to_array(image, IMAGE_SIZE)
    features = extract_image_features(image)
    print('CNN array:', tensor.shape, tensor.dtype, 'range:', (tensor.min(), tensor.max()))
    print('Feature vector:', features.shape, 'finite:', np.isfinite(features).all())
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    axes[0].imshow(image); axes[0].set_title('RGB image')
    axes[1].imshow(tensor); axes[1].set_title('resized [0,1] pixels')
    for ax in axes: ax.axis('off')
    plt.show()

## Baseline descriptors
Features include RGB means/standard deviations and histograms, luminance percentiles, gradient magnitude summaries, and coarse block brightness/variation. They are generic image statistics, not validated forensic signals or manipulation explanations. The CNN uses MobileNetV2's own preprocessing function rather than this [0,1] tensor directly.

In [ ]:
if image is not None:
    names = [f'feature_{i:02d}' for i in range(features.size)]
    list(zip(names[:10], features[:10]))